In [6]:
from __future__ import annotations # postponed evaluation
from dataclasses import dataclass

@dataclass
class Curve:
    """
    Elliptic Curve over the field of integers modula a prime.
    Points on the curve satisfy y^2 = x^2 + a*x + b (mod p).
    """
    p: int
    a: int
    b: int

bitcoin_curve = Curve(
    p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F,
    a = 0x0000000000000000000000000000000000000000000000000000000000000000, # a = 0
    b = 0x0000000000000000000000000000000000000000000000000000000000000007, # b = 7
)

In [8]:
# Define the Generator point
# The generator point is a fixed starting point on the curve's cycle, which is used to kick off the walk around the curve

@dataclass
class Point:
    """
    An integer point (x,y) on a curve.
    """
    curve: Curve
    x: int
    y: int
    
G = Point(
    bitcoin_curve,
    x = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
    y = 0x483ada7726a3c4655da4fbfc0e1108a8fd17b448a68554199c47d08ffb10d4b8,
)

# Verify that the generator point is on the curve
print('Is the generator point on the curve- ', (G.y**2 - G.x**3 - 7) % bitcoin_curve.p == 0)

# Random points will not be on the curve
import random
random.seed(69)
x = random.randrange(0, bitcoin_curve.p)
y = random.randrange(0, bitcoin_curve.p)
print('Is a random point on the curve- ', (y**2 - x**3 - 7) % bitcoin_curve.p == 0)


Is the generator point on the curve-  True
Is a random point on the curve-  False


In [12]:
@dataclass
class Generator:
    """
    A generator over a curve: an initial point and the pre-computed order.
    """
    G: Point # A generator point on the curve
    n: int # The order of the generating point, so 0*G = n*G = INF
        
bitcoin_gen = Generator(
    G = G,
    # The order of G is known and can be mathematically derived
    n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141,
)

In [14]:
# Create private key
secret_key = int.from_bytes(b'RANCH FC. HUMUS', 'big')
assert 1 <= secret_key < bitcoin_gen.n
print(secret_key)

427092899643460065618141604723119443


In [18]:
# The public_key is a point on the curve that results from adding the generator point to itself secret_key times
# public_key = G * secret_key

INF = Point(None, None, None)

def extended_euclidean_algorithm(a, b):
    """
    Returns (gcd, x, y) s.t. a * x + b * y == gcd.
    """
    
    old_r, r = a, b
    old_s, s = 1, 0
    old_t, t = 0, 1
    while r != 0:
        quotient = old_r // r
        old_r, r = r, old_r - quotient * r
        old_s, s = s, old_s - quotient * s
        old_t, t = t, old_t - quotient * t
    return old_r, old_s, old_t

def inv(n, p):
    """
    Returns modular multiplcate inverse m s.t. (n * m) % p == 1.
    """
    gcd, x, y = extended_euclidean_algorithm(n, p)
    return x % p

def elliptic_curve_addition(self, other: Point) -> Point:
    # Handle special case of P + 0 = 0 + P = 0
    if self == INF:
        return other
    if other == INF:
        return self
    # Handle special case of P + (-P) = 0
    if self.x == other.x and self.y != other.y:
        return INF
    # Compute the "slope"
    if self.x == other.x: # (self.y = other.y is guaranteed too per above check)
        m = (3 * self.x**2 + self.curve.a) * inv(2 * self.y, self.curve.p)
    else:
        m = (self.y - other.y) * inv(self.x - other.x, self.curve.p)
    # Compute the new point
    rx = (m**2 - self.x - other.x) % self.curve.p
    ry = (-(m*(rx - self.x) + self.y)) % self.curve.p
    return Point(self.curve, rx, ry)

Point.__add__ = elliptic_curve_addition